# UAS BIG DATA LANJUT
## Analisis & Prediksi Keberhasilan Kampanye Pemasaran Bank

Nama : GABRIEL ANDRA P SOLIN

NIM  : 23.11.5637

Kelas: IF07


**SOAL-1** Studi Kasus: Analisis & Prediksi Keberhasilan Kampanye Pemasaran Bank

Dataset:
🔗 https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing

Studi Kasus & Karakteristik Big Data (5V)

Studi Kasus

Dataset Bank Marketing berisi data hasil kampanye pemasaran bank yang bertujuan
menarik nasabah untuk berlangganan deposito berjangka.
Target utama analisis adalah memprediksi apakah nasabah akan berlangganan deposito (y).

 Big Data 5V (memenuhi ≥ 3)



Volume: Dataset terdiri dari puluhan ribu data nasabah

Variety: Terdapat data numerik dan kategorikal

Velocity: Data dikumpulkan secara berkala dalam aktivitas kampanye

Value: Data bernilai tinggi untuk pengambilan keputusan bisnis bank

Penyimpanan Data (File System)

Dataset disimpan di Google Drive dan diakses melalui Google Colab.
Google Drive digunakan sebagai alternatif distributed file system pengganti HDFS
karena mendukung akses data besar secara terpusat dan terintegrasi dengan Colab.

**SOAL 2: Penyimpanan Data (File System)**

Dataset disimpan di Google Drive dan diakses melalui Google Colab.
Google Drive digunakan sebagai alternatif distributed file system pengganti HDFS karena mendukung akses data besar secara terpusat dan terintegrasi dengan Colab.

**nisialisasi Lingkungan PySpark**

In [20]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Google Drive di-mount agar dataset dapat diakses langsung oleh PySpark
tanpa perlu upload manual

**SOAL 3: Pemrosesan Data Menggunakan PySpark**

**Inisialisasi Spark Session**

In [21]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BankMarketingBigData") \
    .getOrCreate()


SparkSession merupakan entry point utama untuk menggunakan PySpark
dalam pemrosesan big data.

**Load Dataset**

In [22]:
path = "/content/drive/MyDrive/uas BigData lanjut/bank-additional-full.csv"
df = spark.read.csv(path, header=True, inferSchema=True, sep=";")
df.show(5)


+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
|age|      job|marital|  education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|cons.price.idx|cons.conf.idx|euribor3m|nr.employed|  y|
+---+---------+-------+-----------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
| 56|housemaid|married|   basic.4y|     no|     no|  no|telephone|  may|        mon|     261|       1|  999|       0|nonexistent|         1.1|        93.994|        -36.4|    4.857|     5191.0| no|
| 57| services|married|high.school|unknown|     no|  no|telephone|  may|        mon|     149|       1|  999|       0|nonexistent|         1.1|        93.994|        -36.4|    4.857|     5191.0| no|
| 37| serv

Dataset dibaca menggunakan Spark DataFrame dengan:

header=True untuk membaca nama kolom

inferSchema=True untuk mendeteksi tipe data otomatis

**A.Batch Processing (MapReduce)**

In [23]:
rdd = df.select("y").rdd.map(lambda x: (x[0], 1))
hasil = rdd.reduceByKey(lambda a, b: a + b)
hasil.collect()


[('no', 36548), ('yes', 4640)]

Pemrosesan batch dilakukan menggunakan konsep MapReduce:

Map: memetakan nilai target

Reduce: menghitung jumlah masing-masing kelas

**B.Exploratory Data Analysis (EDA)**

Distribusi Target

In [24]:
df.groupBy("y").count().show()


+---+-----+
|  y|count|
+---+-----+
| no|36548|
|yes| 4640|
+---+-----+



Statistik Data Numerik

In [25]:
df.select(
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous"
).describe().show()


+-------+------------------+------------------+------------------+-----------------+-------------------+
|summary|               age|          duration|          campaign|            pdays|           previous|
+-------+------------------+------------------+------------------+-----------------+-------------------+
|  count|             41188|             41188|             41188|            41188|              41188|
|   mean| 40.02406040594348| 258.2850101971448| 2.567592502670681|962.4754540157328|0.17296299893172767|
| stddev|10.421249980934043|259.27924883646494|2.7700135429023245|186.9109073447411| 0.4949010798392903|
|    min|                17|                 0|                 1|                0|                  0|
|    max|                98|              4918|                56|              999|                  7|
+-------+------------------+------------------+------------------+-----------------+-------------------+



Exploratory Data Analysis (EDA) dilakukan pada beberapa atribut numerik,
yaitu age, duration, campaign, pdays, dan previous.
Statistik deskriptif digunakan untuk mengetahui karakteristik data
meliputi nilai minimum, maksimum, rata-rata, serta sebaran data.
Hasil analisis ini digunakan sebagai dasar untuk memahami pola data
sebelum dilakukan tahap preprocessing dan pemodelan machine learning.


**C.Preprocessing Data**

In [27]:
df.printSchema()


root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- emp.var.rate: double (nullable = true)
 |-- cons.price.idx: double (nullable = true)
 |-- cons.conf.idx: double (nullable = true)
 |-- euribor3m: double (nullable = true)
 |-- nr.employed: double (nullable = true)
 |-- y: string (nullable = true)



Pengecekan struktur dan tipe data dilakukan untuk mengetahui
format setiap atribut pada dataset. Informasi ini penting
sebagai dasar dalam menentukan proses preprocessing selanjutnya.


Cek Missing Value (STABIL & ANTI ERROR)
KODE

In [28]:
df.selectExpr(
    "count(*) as total_rows",
    "count(age) as age_non_null",
    "count(duration) as duration_non_null",
    "count(campaign) as campaign_non_null",
    "count(pdays) as pdays_non_null",
    "count(previous) as previous_non_null"
).show()


+----------+------------+-----------------+-----------------+--------------+-----------------+
|total_rows|age_non_null|duration_non_null|campaign_non_null|pdays_non_null|previous_non_null|
+----------+------------+-----------------+-----------------+--------------+-----------------+
|     41188|       41188|            41188|            41188|         41188|            41188|
+----------+------------+-----------------+-----------------+--------------+-----------------+



Pengecekan missing value dilakukan dengan membandingkan jumlah total baris
dan jumlah nilai non-null pada setiap atribut numerik.
Hasil menunjukkan seluruh atribut memiliki nilai lengkap,
sehingga tidak diperlukan penanganan missing value.


**Casting Tipe Data Numerik**

In [29]:
from pyspark.sql.functions import col

df = df.withColumn("age", col("age").cast("int")) \
       .withColumn("duration", col("duration").cast("int")) \
       .withColumn("campaign", col("campaign").cast("int")) \
       .withColumn("pdays", col("pdays").cast("int")) \
       .withColumn("previous", col("previous").cast("int"))


Proses casting tipe data dilakukan untuk memastikan seluruh atribut numerik
memiliki format integer agar dapat diproses dengan baik
oleh algoritma machine learning.


**Encoding Target (Label Encoding)**

In [30]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="y",
    outputCol="label"
)

df = label_indexer.fit(df).transform(df)


Kolom target y yang bersifat kategorikal dikonversi menjadi nilai numerik
menggunakan StringIndexer agar dapat digunakan
dalam pemodelan supervised learning.


**Verifikasi Hasil Preprocessing**

In [31]:
df.select("y", "label").show(5)


+---+-----+
|  y|label|
+---+-----+
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
| no|  0.0|
+---+-----+
only showing top 5 rows


Tahap verifikasi dilakukan untuk memastikan bahwa proses encoding
berhasil mengubah label target ke dalam bentuk numerik.


**D.Manipulasi Data (Spark SQL & Agregasi)**

Membuat Temporary View

In [32]:
df.createOrReplaceTempView("bank")


Temporary view dibuat agar dataset dapat diakses menggunakan Spark SQL.
Pendekatan ini memudahkan analisis data berbasis query SQL
seperti agregasi, filtering, dan grouping.


**Query Spark SQL (Agregasi)**

In [33]:
spark.sql("""
SELECT job, COUNT(*) AS total_nasabah
FROM bank
WHERE y = 'yes'
GROUP BY job
ORDER BY total_nasabah DESC
""").show()


+-------------+-------------+
|          job|total_nasabah|
+-------------+-------------+
|       admin.|         1352|
|   technician|          730|
|  blue-collar|          638|
|      retired|          434|
|   management|          328|
|     services|          323|
|      student|          275|
|self-employed|          149|
|   unemployed|          144|
| entrepreneur|          124|
|    housemaid|          106|
|      unknown|           37|
+-------------+-------------+



Query Spark SQL digunakan untuk menganalisis jumlah nasabah
yang berlangganan deposito berdasarkan jenis pekerjaan.
Hasil agregasi ini memberikan insight bisnis
terkait segmen pekerjaan yang paling responsif terhadap kampanye.


**Operasi RDD & MapReduce Lanjutan**

In [34]:
job_rdd = df.select("job").rdd.map(lambda x: (x[0], 1))
job_rdd.reduceByKey(lambda a, b: a + b).collect()


[('retired', 1720),
 ('management', 2924),
 ('unemployed', 1014),
 ('self-employed', 1421),
 ('unknown', 330),
 ('housemaid', 1060),
 ('services', 3969),
 ('admin.', 10422),
 ('blue-collar', 9254),
 ('technician', 6743),
 ('entrepreneur', 1456),
 ('student', 875)]

Operasi RDD dilakukan menggunakan fungsi map dan reduceByKey
untuk menghitung jumlah data berdasarkan kategori pekerjaan.
Pendekatan ini menunjukkan penggunaan pemrosesan terdistribusi
berbasis RDD dalam PySpark.


**E.Feature Engineering**

In [35]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["age", "duration", "campaign", "pdays", "previous"],
    outputCol="features"
)

df = assembler.transform(df)


Feature engineering dilakukan dengan menggabungkan beberapa
atribut numerik ke dalam satu vektor fitur.
Langkah ini diperlukan agar data dapat digunakan
oleh algoritma machine learning di MLlib.


**SOAL 4: Pemodelan Machine Learning (MLlib)**

In [36]:
train, test = df.randomSplit([0.8, 0.2], seed=42)


Dataset dibagi menjadi data latih dan data uji
dengan rasio 80:20 untuk mengevaluasi performa model secara objektif.


**Model 1 – Logistic Regression**

In [37]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

lr_model = lr.fit(train)


**Model 2 – Decision Tree**

In [38]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

dt_model = dt.fit(train)


Dua algoritma supervised learning yaitu Logistic Regression
dan Decision Tree digunakan untuk membandingkan performa model
dalam memprediksi keberhasilan kampanye pemasaran bank.


**SOAL 5: Hyperparameter Tuning**

In [40]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.maxIter, [10, 20]) \
    .build()

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

cv_model = cv.fit(train)


Hyperparameter tuning dilakukan menggunakan Cross Validation
untuk memperoleh konfigurasi parameter terbaik
pada model Logistic Regression.


**SOAL 6: Evaluasi ModelEvaluasi Model**

valuasi AUC

In [39]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

print("AUC Logistic Regression:",
      evaluator.evaluate(lr_model.transform(test)))

print("AUC Decision Tree:",
      evaluator.evaluate(dt_model.transform(test)))


AUC Logistic Regression: 0.855536366032948
AUC Decision Tree: 0.3132855826686353


Evaluasi model dilakukan menggunakan metrik Area Under Curve (AUC)
karena permasalahan bersifat klasifikasi biner.
Nilai AUC digunakan untuk menentukan model terbaik.


**Kesimpulan Akhir**

Berdasarkan hasil eksperimen, model Logistic Regression
dengan hyperparameter tuning memberikan performa terbaik
dalam memprediksi keberhasilan kampanye pemasaran bank.
Pendekatan Big Data menggunakan PySpark terbukti mampu
mengolah data berskala besar secara efisien dan akurat.

